In [1]:
from pydantic import BaseModel
import json
import pandas as pd
from pydantic import ValidationError
from pandas import DataFrame
from ollama import generate
from transformers import AutoTokenizer, pipeline

/home/daisy/konema/Documents/partages/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class MCQQuestion(BaseModel):
    question1: str
    option_a1: str
    option_b1: str
    option_c1: str
    option_d1: str
    correct_option1: str
    question2: str
    option_a2: str
    option_b2: str
    option_c2: str
    option_d2: str
    correct_option2: str

In [3]:
def validate_mcq(mcq_json):
    try:
        return MCQQuestion.model_validate_json(mcq_json)
    except ValidationError as e:
        print(f"Validation failed: {e}")
        return None
        


def flatten_and_export_mcq(df: DataFrame, export_filename: str, mcq_column_name: str):
    ids = [x for val in df["id"] for x in (val, val+'-') ]
    result_df = pd.DataFrame({"id": ids})
    
    result_df['question'] = pd.concat([df[mcq_column_name].apply(lambda x: x.question1 if x else ""), df[mcq_column_name].apply(lambda x: x.question2 if x else "")], ignore_index=True)
    result_df['option_a'] = pd.concat([df[mcq_column_name].apply(lambda x: x.option_a1 if x else ""), df[mcq_column_name].apply(lambda x: x.option_a2 if x else "")], ignore_index=True)
    result_df['option_b'] = pd.concat([df[mcq_column_name].apply(lambda x: x.option_b1 if x else ""), df[mcq_column_name].apply(lambda x: x.option_b2 if x else "")], ignore_index=True) 
    result_df['option_c'] = pd.concat([df[mcq_column_name].apply(lambda x: x.option_c1 if x else ""), df[mcq_column_name].apply(lambda x: x.option_c2 if x else "")], ignore_index=True)
    result_df['option_d'] = pd.concat([df[mcq_column_name].apply(lambda x: x.option_d1 if x else ""), df[mcq_column_name].apply(lambda x: x.option_d2 if x else "")], ignore_index=True)
    result_df['correct_option'] = pd.concat([df[mcq_column_name].apply(lambda x: x.correct_option1 if x else ""),df[mcq_column_name].apply(lambda x: x.correct_option2 if x else "")],ignore_index=True)
    
    result_df.to_csv(export_filename, index=False)

In [4]:

def generate_mcq(content, model_name, temperature):
    prompt = f"""
        À partir du contenu éducatif suivant, générez deux questions à choix multiple avec quatre options de réponse dont une seule est correcte.
        La question doit évaluer la compréhension des idées principales, et les options doivent être claires, informatives et pertinentes.
        Assurez-vous que les distracteurs (options incorrectes) suivent une interprétation logique mais incorrecte, basée sur des idées reçues ou des incompréhensions courantes du sujet.
        Les options de réponse doivent être aussi courtes que possible.

        IMPORTANT — FORMAT ABSOLU POUR LES CHAMPS 'correct_option1' ET 'correct_option2' :
        - Ces champs doivent contenir exactement **une seule lettre minuscule** parmi : a, b, c ou d.
        - **Exemples valides** : "a", "b", "c", "d".
        - **Interdits** : "a)", "A", "a.", "a )", "le texte de la réponse correcte", 1, true, etc.
        - La sortie JSON doit conserver ces champs comme chaînes (`"correct_option1": "a"`).

        Fournissez la sortie strictement au format JSON correspondant au schéma demandé (ne pas produire de texte hors-du-JSON).
        **Contenu éducatif :**
        {content}
    """
    
    generate_params = {
        'model': model_name,
        'options': {'temperature': temperature, 'num_ctx': 8192, 'top_p': 1}, 
        'prompt': prompt,
        'format': MCQQuestion.model_json_schema()
    }
    
    # Get a response
    response = generate(**generate_params)
    return response['response']

In [5]:
def generate_mcq_hf(content, model_name, temperature):
    prompt = f"""
        À partir du contenu éducatif suivant, générez deux questions à choix multiple avec quatre options de réponse dont une seule est correcte.
        La question doit évaluer la compréhension des idées principales, et les options doivent être claires, informatives et pertinentes.
        Assurez-vous que les distracteurs (options incorrectes) suivent une interprétation logique mais incorrecte, basée sur des idées reçues ou des incompréhensions courantes du sujet.
        Les options de réponse doivent être aussi courtes que possible.
        IMPORTANT — FORMAT ABSOLU POUR LES CHAMPS 'correct_option1' ET 'correct_option2' :
        - Ces champs doivent contenir exactement **une seule lettre minuscule** parmi : a, b, c ou d.
        - **Exemples valides** : "a", "b", "c", "d".
        - **Interdits** : "a)", "A", "a.", "a )", "le texte de la réponse correcte", 1, true, etc.
        - La sortie JSON doit conserver ces champs comme chaînes `"correct_option1": "a"`).
        Fournissez la sortie strictement au format JSON correspondant au schéma demandé (ne pas produire de texte hors-du-JSON).
        **Contenu éducatif :**
        {content}
        Répondez UNIQUEMENT avec le JSON selon ce schéma : {MCQQuestion.model_json_schema()}
    """
    
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        use_fast=True,
        trust_remote_code=True
    )
    
    pipe = pipeline(
        "text-generation",
        model=model_name,
        tokenizer=tokenizer,
        device_map="auto"
    )
    
    messages = [{"role": "user", "content": prompt}]
    
    response = pipe(
        messages,
        max_new_tokens=2048,
        temperature=temperature,
        top_p=1.0,
        do_sample=True,
        return_full_text=False
    )
    
    return response[0]['generated_text']

In [6]:
df = pd.read_csv("../data/lisa_sheets.csv")

In [7]:
file_path = "../data/train_test_split/test_folders.json"

In [8]:
with open(file_path, "r", encoding="utf-8") as file:
    test_folders = json.load(file)

In [9]:
df_test = df[df.folder.isin(test_folders)]
print("Number of lisa sheets :", len(df_test))

Number of lisa sheets : 1592


In [ ]:
df_test['generated_llama3_1_8b'] = df_test['content_raw'].apply(
    lambda content: generate_mcq(content, model_name="linbeiJiang/Llama-3.1-8B-Instruct", temperature=0.1)
)

df_test["llama3_1_8b"] = df_test['generated_llama3_1_8b'].apply(validate_mcq)
flatten_and_export_mcq(df_test, '../data/base_models/instruct/llama3_1_8b.csv', 'llama3_1_8b')

CPU times: user 6.94 s, sys: 427 ms, total: 7.37 s
Wall time: 3h 46min 54s


<timed exec>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
<timed exec>:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


In [ ]:
df_test['generated_llama3_1_8b'] = df_test['content_raw'].apply(
    lambda content: generate_mcq(content, model_name="llama3.1:8b", temperature=0.1)
)

df_test["llama3_1_8b"] = df_test['generated_llama3_1_8b'].apply(validate_mcq)
flatten_and_export_mcq(df_test, '../data/base_models/llama3_1_8b.csv', 'llama3_1_8b')

CPU times: user 4.12 s, sys: 249 ms, total: 4.37 s
Wall time: 55min 40s


<timed exec>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
<timed exec>:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


In [ ]:
df_test['generated_openbiollm_8b'] = df_test['content_raw'].apply(
    lambda content: generate_mcq(content, model_name="koesn/llama3-openbiollm-8b:latest", temperature=0.1)
)

df_test["openbiollm_8b"] = df_test['generated_openbiollm_8b'].apply(validate_mcq)
flatten_and_export_mcq(df_test, '../data/base_models/openbiollm_8b.csv', 'openbiollm_8b')

CPU times: user 4.22 s, sys: 371 ms, total: 4.59 s
Wall time: 55min 31s


<timed exec>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
<timed exec>:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


In [ ]:
df_test['generated_gemma2_9b'] = df_test['content_raw'].apply(
    lambda content: generate_mcq(content, model_name="gemma2:9b", temperature=0.1)
)

df_test["gemma2_9b"] = df_test['generated_gemma2_9b'].apply(validate_mcq)
flatten_and_export_mcq(df_test, '../data/base_models/gemma2_9b.csv', 'gemma2_9b')

CPU times: user 4.07 s, sys: 222 ms, total: 4.29 s
Wall time: 1h 9min 2s


<timed exec>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
<timed exec>:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


In [ ]:
df_test['generated_medGemma_4b'] = df_test['content_raw'].apply(
    lambda content: generate_mcq(content, model_name="alibayram/medgemma:4b", temperature=0.1)
)

df_test["medGemma_4b"] = df_test['generated_medGemma_4b'].apply(validate_mcq)
flatten_and_export_mcq(df_test, '../data/base_models/medGemma_4b.csv', 'medGemma_4b')

Validation failed: 1 validation error for MCQQuestion
  Invalid JSON: EOF while parsing a string at line 13 column 318945 [type=json_invalid, input_value='{\n  "question1": "Selon...usage nocif, pas de l\'', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/json_invalid
Validation failed: 1 validation error for MCQQuestion
  Invalid JSON: EOF while parsing a string at line 4 column 365221 [type=json_invalid, input_value='{\n"question1": "Dans le...use non dégénérative', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/json_invalid
Validation failed: 1 validation error for MCQQuestion
  Invalid JSON: EOF while parsing a string at line 8 column 341304 [type=json_invalid, input_value='{\n"question1": "Quel es...et éclairé, et de s\'', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/json_invalid
Validation failed: 1 validation error for MCQQuestion
  Invalid JSON: EOF while parsing a

<timed exec>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
<timed exec>:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


In [ ]:
df_test['generated_medGemma_27b'] = df_test['content_raw'].apply(
    lambda content: generate_mcq(content, model_name="alibayram/medgemma:27b", temperature=0.1)
)

df_test["medGemma_27b"] = df_test['generated_medGemma_27b'].apply(validate_mcq)
flatten_and_export_mcq(df_test, '../data/base_models/medGemma_27b.csv', 'medGemma_27b')

CPU times: user 4.65 s, sys: 482 ms, total: 5.14 s
Wall time: 2h 59min 54s


<timed exec>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
<timed exec>:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


In [33]:
df_test['generated_qwen3'] = df_test['content_raw'][:5].apply(
    lambda content: generate_mcq_hf(content, model_name="Qwen/Qwen3-0.6B", temperature=0.1)
)


Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0
/tmp/ipykernel_97909/3341846560.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test['generated_qwen3'] = df_test['content_raw'][:5].apply(


In [37]:
df_test.head()

,folder,id,content_raw,rubric,generated_mistral_7B,mistral_7B,generated_qwen3,qwen3
99,IC-005,OIC-005-01-A,{{objectif de connaissance\n|Identifiant=OIC-0...,Définition,"<think>\nOkay, let's tackle this. The user wan...",None,"<think>\nOkay, let's tackle this. The user wan...",None
100,IC-005,OIC-005-02-A,{{objectif de connaissance\n|Identifiant=OIC-0...,Définition,"<think>\nOkay, let's tackle this. The user wan...",None,"<think>\nOkay, let's tackle this. The user wan...",None
101,IC-005,OIC-005-03-A,{{objectif de connaissance\n|Identifiant=OIC-0...,Définition,"<think>\nOkay, let's tackle this. The user wan...",None,"<think>\nOkay, let's tackle this. The user wan...",None
102,IC-005,OIC-005-04-A,{{objectif de connaissance\n|Identifiant=OIC-0...,Définition,"<think>\nOkay, let's tackle this. The user wan...",None,"<think>\nOkay, let's tackle this. The user wan...",None
103,IC-005,OIC-005-05-A,{{objectif de connaissance\n|Identifiant=OIC-0...,Définition,"<think>\nOkay, let's tackle this. The user wan...",None,"<think>\nOkay, let's tackle this. The user wan...",None


In [34]:
df_test["qwen3"] = df_test['generated_qwen3'][:5].apply(validate_mcq)
flatten_and_export_mcq(df_test, '../data/base_models/instruct/qwen3.csv', 'qwen3')

Validation failed: 1 validation error for MCQQuestion
  Invalid JSON: expected value at line 1 column 1 [type=json_invalid, input_value='<think>\nOkay, let\'s ta...n2": "d"\n    }\n  }\n}', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/json_invalid
Validation failed: 1 validation error for MCQQuestion
  Invalid JSON: expected value at line 1 column 1 [type=json_invalid, input_value='<think>\nOkay, let\'s ta..."a"\n    }\n  }\n}\n```', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/json_invalid
Validation failed: 1 validation error for MCQQuestion
  Invalid JSON: expected value at line 1 column 1 [type=json_invalid, input_value='<think>\nOkay, let\'s ta...n      }\n    }\n  }\n}', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/json_invalid
Validation failed: 1 validation error for MCQQuestion
  Invalid JSON: expected value at line 1 column 1 [type=json_invalid, input_value=

/tmp/ipykernel_97909/3266013776.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["qwen3"] = df_test['generated_qwen3'][:5].apply(validate_mcq)


AttributeError: 'float' object has no attribute 'question1'